# AIOps record-level features

Builds record-level PyTorch features for each layer.


In [ ]:
from functools import reduce
import json

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog_name", "hant-catalog")
dbutils.widgets.text("schema_name", "hsl")
dbutils.widgets.dropdown("persist_reports", "true", ["true", "false"])
dbutils.widgets.text("storage_account", "streanmingdatasta")
dbutils.widgets.text("lakehouse_container", "lakehouse")
dbutils.widgets.text("record_feature_base_path", "")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
PERSIST_REPORTS = dbutils.widgets.get("persist_reports").lower() == "true"
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
LAKEHOUSE_CONTAINER = dbutils.widgets.get("lakehouse_container")
RECORD_FEATURE_BASE_PATH_WIDGET = dbutils.widgets.get("record_feature_base_path").strip()
RECORD_FEATURE_BASE_PATH = RECORD_FEATURE_BASE_PATH_WIDGET or f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog/aiops/record_features"

BRONZE_TABLE = "training_bronze_hsl_vehicle_position"
SILVER_TABLE = "training_silver_hsl_vehicle_position"
GOLD_TABLE = "training_gold_hsl_vehicle_position"

UNIFIED_FEATURE_TABLE = "report_aiops_record_features"
BRONZE_FEATURE_TABLE = "report_aiops_bronze_record_features"
SILVER_FEATURE_TABLE = "report_aiops_silver_record_features"
GOLD_FEATURE_TABLE = "report_aiops_gold_record_features"


def qname(table_name: str) -> str:
    return f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{table_name}"


def table_exists(table_name: str) -> bool:
    try:
        spark.table(qname(table_name)).limit(1).collect()
        return True
    except Exception:
        return False


def read_table(table_name: str):
    return spark.table(qname(table_name)) if table_exists(table_name) else None


def save_report(df: DataFrame, table_name: str, partition_cols: list[str] | None = None) -> None:
    if not PERSIST_REPORTS:
        return
    path = f"{RECORD_FEATURE_BASE_PATH.rstrip('/')}/{table_name}"
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    spark.sql(f"DROP TABLE IF EXISTS {qname(table_name)}")
    spark.sql(f"CREATE TABLE {qname(table_name)} USING DELTA LOCATION '{path}'")
    spark.sql(f"REFRESH TABLE {qname(table_name)}")
    print(json.dumps({"saved_report_table": qname(table_name), "path": path}, default=str))


print(json.dumps({
    "bronze_source": qname(BRONZE_TABLE),
    "silver_source": qname(SILVER_TABLE),
    "gold_source": qname(GOLD_TABLE),
    "record_feature_base_path": RECORD_FEATURE_BASE_PATH,
}, indent=2))


In [ ]:
TRANSPORT_MODES = ["bus", "tram", "train", "metro", "ferry", "ubus", "robot"]
DELAY_STATUSES = ["early", "on_time", "delayed", "severely_delayed", "unknown"]
OCCUPANCY_STATUSES = ["empty_or_low", "moderate", "busy", "crowded", "unknown"]
LOCATION_QUALITIES = ["ok", "missing", "out_of_bounds"]
LOCATION_SOURCES = ["GPS", "ODO", "MAN", "DR", "N/A"]


def flag(condition):
    return F.when(condition, F.lit(1.0)).otherwise(F.lit(0.0))


def present_flag(col_name: str):
    return flag(F.col(col_name).isNotNull() & (F.trim(F.col(col_name).cast("string")) != ""))


def null_or_empty_flag(col_name: str):
    return flag(F.col(col_name).isNull() | (F.trim(F.col(col_name).cast("string")) == ""))


def clipped_double(col_name: str, min_value: float, max_value: float):
    return (
        F.when(F.col(col_name).isNull(), F.lit(0.0))
        .when(F.col(col_name).cast("double") < F.lit(min_value), F.lit(min_value))
        .when(F.col(col_name).cast("double") > F.lit(max_value), F.lit(max_value))
        .otherwise(F.col(col_name).cast("double"))
    )


def source_type_for_rule(rule_id: str) -> str:
    r = rule_id.lower()
    if any(x in r for x in ["parse", "json", "mqtt", "event_type", "transport_mode", "format"]):
        return "schema_parse"
    if any(x in r for x in ["null", "missing", "present"]):
        return "completeness"
    if any(x in r for x in ["out_of_bounds", "negative", "range", "invalid", "unexpected"]):
        return "validity"
    if any(x in r for x in ["mismatch", "duplicate", "unique"]):
        return "consistency"
    if any(x in r for x in ["future", "stale", "timestamp", "unix", "delay"]):
        return "timeliness"
    return "other"


def rule_metadata(rule_specs: list[dict]) -> str:
    rows = []
    for spec in rule_specs:
        rows.append({
            "rule_id": spec["rule_id"],
            "severity": spec["severity"],
            "source_type": source_type_for_rule(spec["rule_id"]),
            "feature_column": f"rule_{spec['rule_id']}_flag",
        })
    return json.dumps(rows)


def add_rule_flags(df: DataFrame, rule_specs: list[dict]) -> DataFrame:
    for spec in rule_specs:
        df = df.withColumn(f"rule_{spec['rule_id']}_flag", flag(spec["condition"]))
    critical_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "critical"]
    high_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "high"]
    medium_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "medium"]
    low_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "low"]

    def sum_cols(cols):
        if not cols:
            return F.lit(0.0)
        return reduce(lambda a, b: a + b, [F.col(c) for c in cols])

    all_rule_cols = critical_cols + high_cols + medium_cols + low_cols
    return (
        df
        .withColumn("critical_rule_fail_count", sum_cols(critical_cols))
        .withColumn("high_rule_fail_count", sum_cols(high_cols))
        .withColumn("medium_rule_fail_count", sum_cols(medium_cols))
        .withColumn("low_rule_fail_count", sum_cols(low_cols))
        .withColumn("total_rule_fail_count", sum_cols(all_rule_cols))
        .withColumn("has_rule_failure_flag", flag(F.col("total_rule_fail_count") > 0))
        .withColumn(
            "max_rule_severity_rank",
            F.when(F.col("critical_rule_fail_count") > 0, F.lit(4.0))
            .when(F.col("high_rule_fail_count") > 0, F.lit(3.0))
            .when(F.col("medium_rule_fail_count") > 0, F.lit(2.0))
            .when(F.col("low_rule_fail_count") > 0, F.lit(1.0))
            .otherwise(F.lit(0.0)),
        )
        .withColumn("rule_metadata_json", F.lit(rule_metadata(rule_specs)))
    )


In [ ]:
def build_bronze_record_features() -> DataFrame | None:
    df = read_table(BRONZE_TABLE)
    if df is None:
        print(f"Skipping missing source table: {BRONZE_TABLE}")
        return None

    duplicate_w = Window.partitionBy("partition", "offset")
    base = (
        df
        .withColumn("layer", F.lit("bronze"))
        .withColumn("business_key", F.lit(None).cast("string"))
        .withColumn("record_id", F.concat_ws("|", F.coalesce(F.col("topic"), F.lit("")), F.col("partition").cast("string"), F.col("offset").cast("string")))
        .withColumn("event_ts", F.col("bronze_ingest_ts"))
        .withColumn("feature_date", F.to_date("bronze_ingest_ts"))
        .withColumn("source_table", F.lit(qname(BRONZE_TABLE)))
        .withColumn("duplicate_partition_offset_count", F.count("*").over(duplicate_w).cast("double"))
        .withColumn("enqueue_to_bronze_delay_sec", (F.col("bronze_ingest_ts").cast("long") - F.col("eventhub_enqueued_ts").cast("long")).cast("double"))
    )

    rule_specs = [
        {"rule_id": "critical_topic_null", "severity": "critical", "condition": F.col("topic").isNull()},
        {"rule_id": "critical_partition_null", "severity": "critical", "condition": F.col("partition").isNull()},
        {"rule_id": "critical_offset_null", "severity": "critical", "condition": F.col("offset").isNull()},
        {"rule_id": "critical_eventhub_enqueued_ts_null", "severity": "critical", "condition": F.col("eventhub_enqueued_ts").isNull()},
        {"rule_id": "critical_raw_json_null", "severity": "critical", "condition": F.col("raw_json").isNull()},
        {"rule_id": "critical_bronze_ingest_ts_null", "severity": "critical", "condition": F.col("bronze_ingest_ts").isNull()},
        {"rule_id": "critical_parse_ok_false", "severity": "critical", "condition": F.coalesce(F.col("parse_ok"), F.lit(False)) == F.lit(False)},
        {"rule_id": "critical_duplicate_partition_offset", "severity": "critical", "condition": F.col("duplicate_partition_offset_count") > 1},
        {"rule_id": "high_parse_error_present", "severity": "high", "condition": F.col("parse_error").isNotNull()},
        {"rule_id": "high_source_not_hsl_hfp_mqtt", "severity": "high", "condition": F.coalesce(F.col("source"), F.lit("")) != F.lit("hsl_hfp_mqtt")},
        {"rule_id": "high_event_type_not_vp", "severity": "high", "condition": F.coalesce(F.col("event_type"), F.lit("")) != F.lit("vp")},
        {"rule_id": "high_mqtt_topic_null", "severity": "high", "condition": F.col("mqtt_topic").isNull()},
        {"rule_id": "high_transport_mode_null", "severity": "high", "condition": F.col("transport_mode").isNull()},
        {"rule_id": "medium_producer_ts_bad_format", "severity": "medium", "condition": F.col("producer_ingest_ts_utc").isNotNull() & (~F.col("producer_ingest_ts_utc").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z$"))},
    ]

    return (
        add_rule_flags(base, rule_specs)
        .select(
            "layer", "record_id", "business_key", "event_ts", "feature_date", "source_table", "rule_metadata_json",
            "critical_rule_fail_count", "high_rule_fail_count", "medium_rule_fail_count", "low_rule_fail_count",
            "total_rule_fail_count", "has_rule_failure_flag", "max_rule_severity_rank",
            "duplicate_partition_offset_count", "enqueue_to_bronze_delay_sec",
            *[f"rule_{s['rule_id']}_flag" for s in rule_specs],
        )
    )


In [ ]:
def build_silver_record_features() -> DataFrame | None:
    df = read_table(SILVER_TABLE)
    if df is None:
        print(f"Skipping missing source table: {SILVER_TABLE}")
        return None

    duplicate_w = Window.partitionBy("business_key")
    base = (
        df
        .withColumn("layer", F.lit("silver"))
        .withColumn("record_id", F.coalesce(F.col("business_key"), F.concat_ws("|", F.col("vehicle_id"), F.col("event_ts_unix").cast("string"), F.col("route_id"), F.col("direction_id"))))
        .withColumn("record_event_ts", F.coalesce(F.col("event_ts"), F.col("dedup_event_ts"), F.col("silver_ingest_ts")))
        .withColumn("feature_date", F.to_date(F.col("record_event_ts")))
        .withColumn("source_table", F.lit(qname(SILVER_TABLE)))
        .withColumn("duplicate_business_key_count", F.count("*").over(duplicate_w).cast("double"))
        .withColumn("latitude_clipped", clipped_double("latitude", 59.0, 61.5))
        .withColumn("longitude_clipped", clipped_double("longitude", 23.0, 26.5))
        .withColumn("speed_clipped", clipped_double("speed", 0.0, 50.0))
        .withColumn("heading_clipped", clipped_double("heading", 0.0, 360.0))
        .withColumn("occupancy_clipped", clipped_double("occupancy", 0.0, 100.0))
        .withColumn("event_to_silver_delay_clipped", clipped_double("event_to_silver_delay_sec", -120.0, 300.0))
    )

    rule_specs = [
        {"rule_id": "critical_event_ts_null", "severity": "critical", "condition": F.col("event_ts").isNull()},
        {"rule_id": "critical_event_ts_unix_null", "severity": "critical", "condition": F.col("event_ts_unix").isNull()},
        {"rule_id": "critical_vehicle_id_null", "severity": "critical", "condition": F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == "")},
        {"rule_id": "critical_duplicate_business_key", "severity": "critical", "condition": F.col("duplicate_business_key_count") > 1},
        {"rule_id": "high_route_id_null", "severity": "high", "condition": F.col("route_id").isNull() | (F.trim(F.col("route_id")) == "")},
        {"rule_id": "high_latitude_out_of_bounds", "severity": "high", "condition": F.col("latitude").isNull() | (F.col("latitude") < 59.0) | (F.col("latitude") > 61.5)},
        {"rule_id": "high_longitude_out_of_bounds", "severity": "high", "condition": F.col("longitude").isNull() | (F.col("longitude") < 23.0) | (F.col("longitude") > 26.5)},
        {"rule_id": "high_event_type_unexpected", "severity": "high", "condition": F.col("event_type").isNotNull() & (F.col("event_type") != "vp")},
        {"rule_id": "medium_topic_operator_id_invalid_format", "severity": "medium", "condition": F.col("topic_operator_id").isNotNull() & ~F.col("topic_operator_id").rlike(r"^\d{4}$")},
        {"rule_id": "medium_topic_vehicle_number_invalid_format", "severity": "medium", "condition": F.col("topic_vehicle_number").isNotNull() & ~F.col("topic_vehicle_number").rlike(r"^\d{5}$")},
        {"rule_id": "medium_topic_direction_invalid", "severity": "medium", "condition": F.col("topic_direction_id").isNotNull() & ~F.col("topic_direction_id").isin("1", "2")},
        {"rule_id": "medium_payload_dir_invalid", "severity": "medium", "condition": F.col("dir").isNotNull() & ~F.col("dir").isin("1", "2")},
        {"rule_id": "medium_transport_mode_unexpected", "severity": "medium", "condition": F.col("transport_mode").isNotNull() & ~F.col("transport_mode").isin(*TRANSPORT_MODES)},
        {"rule_id": "medium_negative_speed", "severity": "medium", "condition": F.col("speed").isNotNull() & (F.col("speed") < 0.0)},
        {"rule_id": "medium_speed_out_of_range", "severity": "medium", "condition": F.col("speed").isNotNull() & ((F.col("speed") < 0.0) | (F.col("speed") > 50.0))},
        {"rule_id": "medium_heading_out_of_range", "severity": "medium", "condition": F.col("heading").isNotNull() & ((F.col("heading") < 0) | (F.col("heading") > 360))},
        {"rule_id": "medium_occupancy_out_of_range", "severity": "medium", "condition": F.col("occupancy").isNotNull() & ((F.col("occupancy") < 0) | (F.col("occupancy") > 100))},
        {"rule_id": "medium_door_status_invalid", "severity": "medium", "condition": F.col("door_status").isNotNull() & ~F.col("door_status").isin(0, 1)},
        {"rule_id": "medium_event_ts_raw_invalid_format", "severity": "medium", "condition": F.col("event_ts_raw").isNotNull() & ~F.col("event_ts_raw").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?Z$")},
        {"rule_id": "medium_event_ts_vs_unix_mismatch", "severity": "medium", "condition": F.col("event_ts").isNotNull() & F.col("event_ts_unix").isNotNull() & (F.abs(F.col("event_ts").cast("long") - F.col("event_ts_unix")) > 2)},
        {"rule_id": "medium_event_future_gt_2m", "severity": "medium", "condition": F.col("event_to_silver_delay_sec").isNotNull() & (F.col("event_to_silver_delay_sec") < -120)},
        {"rule_id": "medium_event_stale_gt_5m", "severity": "medium", "condition": F.col("event_to_silver_delay_sec").isNotNull() & (F.col("event_to_silver_delay_sec") > 300)},
        {"rule_id": "low_location_source_unexpected", "severity": "low", "condition": F.col("location_source").isNotNull() & ~F.col("location_source").isin(*LOCATION_SOURCES)},
        {"rule_id": "low_topic_start_time_invalid_format", "severity": "low", "condition": F.col("topic_start_time").isNotNull() & ~F.col("topic_start_time").rlike(r"^(?:[01]\d|2[0-3]):[0-5]\d$")},
        {"rule_id": "low_journey_start_time_invalid_format", "severity": "low", "condition": F.col("journey_start_time").isNotNull() & ~F.col("journey_start_time").rlike(r"^(?:[01]\d|2[0-3]):[0-5]\d$")},
        {"rule_id": "low_topic_route_mismatch", "severity": "low", "condition": F.col("topic_route_id").isNotNull() & F.col("payload_route_id").isNotNull() & (F.col("topic_route_id") != F.col("payload_route_id"))},
        {"rule_id": "low_topic_vehicle_mismatch", "severity": "low", "condition": F.col("topic_vehicle_number_norm").isNotNull() & F.col("vehicle_number").isNotNull() & (F.col("topic_vehicle_number_norm") != F.col("vehicle_number"))},
        {"rule_id": "low_topic_operator_mismatch", "severity": "low", "condition": F.col("topic_operator_id_norm").isNotNull() & F.col("operator_id").isNotNull() & (F.col("topic_operator_id_norm") != F.col("operator_id"))},
        {"rule_id": "low_topic_direction_mismatch", "severity": "low", "condition": F.col("topic_direction_id").isNotNull() & F.col("direction_id").isNotNull() & (F.col("topic_direction_id") != F.col("direction_id"))},
        {"rule_id": "low_topic_start_time_mismatch", "severity": "low", "condition": F.col("topic_start_time").isNotNull() & F.col("journey_start_time").isNotNull() & (F.col("topic_start_time") != F.col("journey_start_time"))},
        {"rule_id": "low_operating_day_vs_event_date_mismatch", "severity": "low", "condition": F.col("operating_day").isNotNull() & F.col("event_ts").isNotNull() & (F.col("operating_day") != F.to_date(F.col("event_ts")))},
    ]

    return (
        add_rule_flags(base, rule_specs)
        .withColumn("event_ts", F.col("record_event_ts"))
        .select(
            "layer", "record_id", "business_key", "event_ts", "feature_date", "source_table", "rule_metadata_json",
            "critical_rule_fail_count", "high_rule_fail_count", "medium_rule_fail_count", "low_rule_fail_count",
            "total_rule_fail_count", "has_rule_failure_flag", "max_rule_severity_rank",
            "duplicate_business_key_count", "latitude_clipped", "longitude_clipped", "speed_clipped",
            "heading_clipped", "occupancy_clipped", "event_to_silver_delay_clipped",
            *[f"rule_{s['rule_id']}_flag" for s in rule_specs],
        )
    )


In [ ]:
def build_gold_record_features() -> DataFrame | None:
    df = read_table(GOLD_TABLE)
    if df is None:
        print(f"Skipping missing source table: {GOLD_TABLE}")
        return None

    duplicate_w = Window.partitionBy("business_key")
    base = (
        df
        .withColumn("layer", F.lit("gold"))
        .withColumn("record_id", F.coalesce(F.col("business_key"), F.concat_ws("|", F.col("vehicle_id"), F.col("gold_service_ts").cast("string"), F.col("canonical_route_id"), F.col("canonical_direction_id"))))
        .withColumn("record_event_ts", F.coalesce(F.col("gold_service_ts"), F.col("event_ts"), F.col("gold_publish_ts")))
        .withColumn("feature_date", F.to_date(F.col("record_event_ts")))
        .withColumn("source_table", F.lit(qname(GOLD_TABLE)))
        .withColumn("duplicate_business_key_count", F.count("*").over(duplicate_w).cast("double"))
        .withColumn("latitude_clipped", clipped_double("latitude", 59.0, 61.5))
        .withColumn("longitude_clipped", clipped_double("longitude", 23.0, 26.5))
        .withColumn("speed_clipped", clipped_double("speed", 0.0, 50.0))
        .withColumn("occupancy_clipped", clipped_double("occupancy", 0.0, 100.0))
    )

    rule_specs = [
        {"rule_id": "critical_event_ts_null", "severity": "critical", "condition": F.col("event_ts").isNull()},
        {"rule_id": "critical_vehicle_id_null", "severity": "critical", "condition": F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == "")},
        {"rule_id": "critical_business_key_null", "severity": "critical", "condition": F.col("business_key").isNull() | (F.trim(F.col("business_key")) == "")},
        {"rule_id": "critical_duplicate_business_key", "severity": "critical", "condition": F.col("duplicate_business_key_count") > 1},
        {"rule_id": "high_route_id_null", "severity": "high", "condition": F.col("canonical_route_id").isNull() | (F.trim(F.col("canonical_route_id")) == "")},
        {"rule_id": "high_direction_invalid", "severity": "high", "condition": F.col("canonical_direction_id").isNull() | ~F.col("canonical_direction_id").isin("1", "2")},
        {"rule_id": "high_service_date_null", "severity": "high", "condition": F.col("service_date").isNull()},
        {"rule_id": "high_coordinates_out_of_bounds", "severity": "high", "condition": (F.col("latitude").isNotNull() & ((F.col("latitude") < 59.0) | (F.col("latitude") > 61.5))) | (F.col("longitude").isNotNull() & ((F.col("longitude") < 23.0) | (F.col("longitude") > 26.5)))},
        {"rule_id": "medium_transport_mode_unexpected", "severity": "medium", "condition": F.col("transport_mode").isNotNull() & ~F.col("transport_mode").isin(*TRANSPORT_MODES)},
        {"rule_id": "medium_delay_status_invalid", "severity": "medium", "condition": F.col("delay_status").isNull() | ~F.col("delay_status").isin(*DELAY_STATUSES)},
        {"rule_id": "medium_occupancy_status_invalid", "severity": "medium", "condition": F.col("occupancy_status").isNull() | ~F.col("occupancy_status").isin(*OCCUPANCY_STATUSES)},
        {"rule_id": "medium_location_quality_invalid", "severity": "medium", "condition": F.col("location_quality").isNull() | ~F.col("location_quality").isin(*LOCATION_QUALITIES)},
        {"rule_id": "medium_negative_speed", "severity": "medium", "condition": F.col("speed").isNotNull() & (F.col("speed") < 0.0)},
        {"rule_id": "medium_speed_out_of_range", "severity": "medium", "condition": F.col("speed").isNotNull() & ((F.col("speed") < 0.0) | (F.col("speed") > 50.0))},
        {"rule_id": "medium_occupancy_out_of_range", "severity": "medium", "condition": F.col("occupancy").isNotNull() & ((F.col("occupancy") < 0) | (F.col("occupancy") > 100))},
        {"rule_id": "low_location_quality_mismatch", "severity": "low", "condition": (F.col("has_coordinates") & (F.col("location_quality") == "missing")) | (~F.col("has_coordinates") & (F.col("location_quality") == "ok"))},
        {"rule_id": "low_delay_flag_mismatch", "severity": "low", "condition": F.col("delay_sec").isNotNull() & (F.col("is_delayed") != (F.col("delay_sec") > 120))},
        {"rule_id": "low_route_id_canonical_mismatch", "severity": "low", "condition": F.col("route_id").isNotNull() & F.col("canonical_route_id").isNotNull() & (F.col("route_id") != F.col("canonical_route_id"))},
        {"rule_id": "low_direction_id_canonical_mismatch", "severity": "low", "condition": F.col("direction_id").isNotNull() & F.col("canonical_direction_id").isNotNull() & (F.col("direction_id") != F.col("canonical_direction_id"))},
    ]

    return (
        add_rule_flags(base, rule_specs)
        .withColumn("event_ts", F.col("record_event_ts"))
        .select(
            "layer", "record_id", "business_key", "event_ts", "feature_date", "source_table", "rule_metadata_json",
            "critical_rule_fail_count", "high_rule_fail_count", "medium_rule_fail_count", "low_rule_fail_count",
            "total_rule_fail_count", "has_rule_failure_flag", "max_rule_severity_rank",
            "duplicate_business_key_count", "latitude_clipped", "longitude_clipped", "speed_clipped",
            *[f"rule_{s['rule_id']}_flag" for s in rule_specs],
        )
    )


In [ ]:
bronze_features = build_bronze_record_features()
silver_features = build_silver_record_features()
gold_features = build_gold_record_features()

if bronze_features is not None:
    bronze_features = bronze_features.withColumn("layer", F.lit("bronze")).withColumn("business_key", F.lit(None).cast("string"))
if silver_features is not None:
    silver_features = silver_features.withColumn("layer", F.lit("silver"))
if gold_features is not None:
    gold_features = gold_features.withColumn("layer", F.lit("gold"))

feature_layers = [df for df in [bronze_features, silver_features, gold_features] if df is not None]
if not feature_layers:
    raise ValueError("No source tables found. Run the non-GE training data branch first.")

all_cols = sorted(set().union(*[set(df.columns) for df in feature_layers]))
aligned_layers = []
for df in feature_layers:
    for c in [c for c in all_cols if c not in df.columns]:
        if c in {"layer", "record_id", "business_key", "source_table", "rule_metadata_json"}:
            df = df.withColumn(c, F.lit(None).cast("string"))
        elif c in {"event_ts"}:
            df = df.withColumn(c, F.lit(None).cast("timestamp"))
        elif c in {"feature_date"}:
            df = df.withColumn(c, F.lit(None).cast("date"))
        else:
            df = df.withColumn(c, F.lit(0.0).cast("double"))
    aligned_layers.append(df.select(*all_cols))

record_features_df = reduce(lambda left, right: left.unionByName(right), aligned_layers)
numeric_feature_cols = [
    c for c, t in record_features_df.dtypes
    if c not in {"layer", "record_id", "business_key", "event_ts", "feature_date", "source_table", "rule_metadata_json"}
    and t in {"double", "float", "int", "bigint", "smallint", "tinyint"}
]
record_features_df = record_features_df.fillna(0.0, subset=numeric_feature_cols)

save_report(record_features_df.where(F.col("layer") == "bronze"), BRONZE_FEATURE_TABLE, ["feature_date"])
save_report(record_features_df.where(F.col("layer") == "silver"), SILVER_FEATURE_TABLE, ["feature_date"])
save_report(record_features_df.where(F.col("layer") == "gold"), GOLD_FEATURE_TABLE, ["feature_date"])
save_report(record_features_df, UNIFIED_FEATURE_TABLE, ["feature_date", "layer"])

display(record_features_df.groupBy("layer").agg(
    F.count("*").alias("record_count"),
    F.sum("has_rule_failure_flag").alias("records_with_ge_equivalent_rule_flags"),
    F.avg("total_rule_fail_count").alias("avg_rule_fail_count"),
))
display(record_features_df.orderBy(F.desc("total_rule_fail_count")))
